# VLTL-Bench 评估 Notebook

本 Notebook 用于在 VLTL-Bench 数据集上评估从自然语言到 LTL 的翻译效果。




In [1]:
import os, json, pathlib, importlib  # 导入通用标准库模块
from pprint import pprint
import pandas as pd  # 数据处理依赖

# 当前 Notebook 所在路径（用于构造相对路径）
NOTEBOOK_DIR = pathlib.Path.cwd()  # 记录当前笔记本目录，方便构造相对路径

In [2]:
# ---- 用户参数 ----
DATASET_DIR = pathlib.Path("VLTL-Bench/train")  # 数据集根目录，需要按实际路径修改

# 每个数据集最多评估的样本数
N_EXAMPLES   = 500  # 每个数据集评估的样本数量上限
# 需要对比的框架列表
FRAMEWORKS   = ["NL2TL"]   # 需要对比的框架列表，可扩展为 "nl2spec" 等

In [3]:
# ---- 定位数据集文件 ----
if not DATASET_DIR.exists():
    raise FileNotFoundError(DATASET_DIR)  # 友好提示路径不存在

dataset_files = sorted(p for p in DATASET_DIR.glob("*.jsonl"))  # 仅抓取 jsonl 数据集
print(f"Found {len(dataset_files)} datasets:")
for p in dataset_files:
    print("  -", p.name)

Found 3 datasets:
  - search_and_rescue.jsonl
  - traffic_light.jsonl
  - warehouse.jsonl


In [1]:
import os
import json
import pandas as pd
import difflib
from IPython.display import display


# 精确对应 和 最长的连续匹配子序列相似度评估 指标1和2

# ─── 配置 ──────────────────────────────────────────────────────────────
root = "translation_eval/nl2tl"  # nl2tl 评估结果根目录
eval_types = ["Qwen3-4B_r16_a32_lr5e-5", "Qwen3-lifted-vllm", "raw_nl"]  # 不同评估设置
max_entries = 5000  # 样本数量限制
# ─────────────────────────────────────────────────────────────────────────

# 从第一个 eval_type 目录中推断数据集名称
datasets = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(os.path.join(root, eval_types[0]))
    if f.endswith(".jsonl")
])

# 初始化结果表
acc_df = pd.DataFrame(index=eval_types, columns=datasets, dtype=float)  # 精确匹配率表
sim_df = pd.DataFrame(index=eval_types, columns=datasets, dtype=float)  # 子串相似度表

def best_substring_similarity(prediction: str, target: str) -> float:
    """
    计算 target 与 prediction 任意等长子串之间的最大 SequenceMatcher 相似度。
    若 prediction 比 target 短，则直接比较完整字符串。
    返回 prediction 中“最像 target 的那一段”的相似度。
    """
    sm = difflib.SequenceMatcher
    t_len, p_len = len(target), len(prediction)
    if p_len < t_len:
        return sm(None, prediction, target).ratio()
    best = 0.0
    for i in range(p_len - t_len + 1):
        sub = prediction[i : i + t_len]
        best = max(best, sm(None, sub, target).ratio())
    return best

# 计算各 eval_type 的指标
for et in eval_types:
    et_dir = os.path.join(root, et)
    for ds in datasets:
        file_path = os.path.join(et_dir, f"{ds}.jsonl")
        if not os.path.isfile(file_path):
            continue

        preds, targets = [], []
        with open(file_path) as f:
            for i, line in enumerate(f):
                if i >= max_entries:
                    break
                ent = json.loads(line)
                # 目标串：优先 masked_tl；否则 raw_nl 情况取 tl
                if "lifted" in et:
                    tgt = " ".join(ent.get("masked_tl", []))
                else:
                    tgt = " ".join(ent.get("tl", []))
                # 清洗预测串（去掉编号前缀）
                pred = ent.get("prediction", "").strip()
                parts = pred.split(" ", 1)
                if parts[0].rstrip(".").isdigit() and len(parts) > 1:
                    pred = parts[1]  # 去掉编号前缀
                preds.append(pred)
                targets.append(tgt)

        if not targets:
            continue

        # 精确匹配率
        acc_df.at[et, ds] = sum(p == t for p, t in zip(preds, targets)) / len(targets)  # 精确匹配率
        # 子串最佳相似度平均
        sim_df.at[et, ds] = sum(
            best_substring_similarity(p, t)
            for p, t in zip(preds, targets)
) / len(targets)  # 子串最佳相似度平均

print("✅ NL2TL Translation Accuracy")
display(sim_df)

✅ NL2TL Translation Accuracy


,search_and_rescue,traffic_light,warehouse
Qwen3-4B_r16_a32_lr5e-5,0.89612,0.920839,0.826149
Qwen3-lifted-vllm,1.00000,1.000000,1.000000
raw_nl,1.00000,1.000000,0.962298


In [2]:
# Jupyter notebook cell: Verification evaluation with parser-error handling
# 该单元用于在存在解析错误时完成形式化验证评估
# 整体流程：
# 1) 将模型输出的自然语言/近似LTL文本标准化为可解析的LTL字符串
# 2) 用 pyModelChecking.LTL.Parser 解析为语法树（AST）
# 3) 在给定的正例/反例轨迹上进行语义求值，统计准确率
# 4) 汇总不同框架/设置的评估结果为 DataFrame

import json
import re
from pathlib import Path
from typing import List, Set, Union
from tqdm import tqdm
from functools import lru_cache
from pyModelChecking.LTL import Parser, AtomicProposition as AP, Not, And, Or, Imply, X, F, G, U
import pandas as pd

# ----------------------------------------------------------------------------
# 1 — Normalisation / implication elimination
# 1 — 规范化与蕴含消除
# ----------------------------------------------------------------------------
# TOKEN_MAP 将常见的英文关键词/符号映射到 LTL 运算符，以容错不同写法
TOKEN_MAP = {
    "globally": "G", "always": "G", "[]": "G",
    "finally": "F", "eventually": "F", "<>": "F",
    "next": "X", "until": "U",
    "not": "not", "¬": "not", "!": "not", 
    "&": "and", "∧": "and",
    "|": "or", "∨": "or", "or": "or",
    "imply": "-->", "implies": "-->", "->": "-->",
    "⇒": "-->",
    "double_implies": "-->"
}
# LTL / TL 公式解析器实例，用于将字符串解析为语法树
_PARSER = Parser()

# 原子命题（Atomic Proposition, AP）的合法命名规则：
# 以字母或下划线开头，后续可包含字母、数字或下划线
# 用于校验 prop 名称或变量名是否合法
_AP_OK = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

def _normalise_tokens(tokens: List[str]) -> str:
    # 将分词后的 token 逐个归一化：关键字映射、括号保留、合法原子保留、非法项加引号
    out = []
    for t in tokens:
        low = t.lower()
        if low in TOKEN_MAP:
            out.append(TOKEN_MAP[low])        # 关键字统一为 LTL 记号
        elif t in ("(", ")"):
            out.append(t)                     # 保留括号
        elif _AP_OK.match(t):
            out.append(t)                     # 合法的原子命题名保留
        else:
            out.append(f"'{t}'")              # 其他内容加引号避免解析冲突
    return " ".join(out)  # 将 token 列表重写为 LTL 友好的字符串

def _elim_impl_tokens(tokens: List[str]) -> List[str]:
    # 消除蕴含：A --> B 重写为 ( (not A) or B )
    # 通过括号层级 depth 识别蕴含两侧的子式范围
    while True:
        depth = [0] * len(tokens)
        d = 0
        for i, tok in enumerate(tokens):
            if tok == "(":
                d += 1
            depth[i] = d
            if tok == ")":
                d -= 1
        for i, tok in enumerate(tokens):
            if tok not in ("->", "-->"):
                continue
            di = depth[i]
            # 回溯找到左侧子式起点
            j = i - 1
            while j >= 0 and depth[j] >= di:
                j -= 1
            lhs_start = j + 1
            # 前进找到右侧子式终点
            k = i + 1
            while k < len(tokens) and depth[k] >= di:
                k += 1
            rhs_end = k
            lhs = tokens[lhs_start:i]
            rhs = tokens[i+1:rhs_end]
            # 用 ( (not LHS) or RHS ) 替代 LHS --> RHS
            new = ["(", "(", "not"] + lhs + [")", "or", "("] + rhs + [")", ")"]
            tokens = tokens[:lhs_start] + new + tokens[rhs_end:]
            break
        else:
            return tokens  # 当没有蕴含符时停止

@lru_cache(maxsize=16384)
def _parse(formula_str: str):
    # 解析并缓存 AST，避免重复解析带来的性能开销
    return _PARSER(formula_str)  # 复用解析结果降低重复解析成本

def _eval(ast, trace: List[Set[str]], t: int = 0) -> bool:
    # LTL 语义求值：在离散轨迹 trace 上从时间 t 开始判断公式真值
    if isinstance(ast, AP):
        return str(ast) in trace[t]                # 原子：检查当前步是否包含该命题
    if isinstance(ast, Not):
        return not _eval(ast.subformula(0), trace, t)
    if isinstance(ast, And):
        return _eval(ast.subformula(0), trace, t) and _eval(ast.subformula(1), trace, t)
    if isinstance(ast, Or):
        return _eval(ast.subformula(0), trace, t) or _eval(ast.subformula(1), trace, t)
    if isinstance(ast, Imply):
        return (not _eval(ast.subformula(0), trace, t)) or _eval(ast.subformula(1), trace, t)
    if isinstance(ast, X):
        return _eval(ast.subformula(0), trace, min(t+1, len(trace)-1))  # X：下一步，越界则取最后一步
    if isinstance(ast, F):
        return any(_eval(ast.subformula(0), trace, k) for k in range(t, len(trace)))  # F：某时成立
    if isinstance(ast, G):
        return all(_eval(ast.subformula(0), trace, k) for k in range(t, len(trace)))  # G：一直成立
    if isinstance(ast, U):
        # 直到：φ U ψ —— 在某个 k 满足 ψ，且从 t..k-1 都满足 φ
        φ, ψ = ast.subformula(0), ast.subformula(1)
        for k in range(t, len(trace)):
            if _eval(ψ, trace, k):
                return all(_eval(φ, trace, j) for j in range(t, k))
        return False
    raise NotImplementedError(f"Unsupported AST node: {type(ast)}")

def _tokenise(tokens: Union[List[str], str]) -> List[str]:
    # 简单分词器：将字符串拆成单词和括号；若已是列表则直接返回
    if isinstance(tokens, str):
        return re.findall(r"\w+|[()]", tokens)  # 简单分词器：拆出单词与括号
    return tokens

# ----------------------------------------------------------------------------
# 2 — Load ground-truth test entries
# 2 — 载入官方测试集用于比对（JSONL，每行一个样本）
# ----------------------------------------------------------------------------
test_set_dir = Path("VLTL-Bench/test")
datasets = ["search_and_rescue", "traffic_light", "warehouse"]
test_entries = {}
for ds in datasets:
    m = {}
    with open(test_set_dir/f"{ds}.jsonl") as f:
        for line in f:
            e = json.loads(line)
            m[e["id"]] = e                    # 用 id 索引，便于快速查找
    test_entries[ds] = m

# ----------------------------------------------------------------------------
# 3 — Run evaluation with parser-error handling
# 3 — 启动评估流程并记录解析错误
# ----------------------------------------------------------------------------
base_eval_dir = Path("translation_eval")
results = []

for fw_dir in base_eval_dir.iterdir():
    if not fw_dir.is_dir(): 
        continue
    framework = fw_dir.name  # 框架名称，如 nl2tl / nl2spec

    # nl2tl structure
    if framework == "nl2tl":
        # 遍历不同 lifting 设置（如 llm_masked_nl / gt_masked_nl / raw_nl）
        for lift_dir in fw_dir.iterdir():
            lifting = lift_dir.name
            for ds in datasets:
                file = lift_dir/f"{ds}.jsonl"
                if not file.exists():
                    continue
                total = ok_good = ok_bad = ok_both = 0
                # 注意：本分支未初始化 bad_parse，下面的 except 使用时可能报错
                for line in tqdm(file.open(), desc=f"{framework}/{lifting}/{ds}"):
                    e = json.loads(line)
                    gt = test_entries[ds][e["id"]]
                    # 将 prop_id -> 原子命题字符串（如 pickup(obj)）的映射重建
                    mapping = {
                        pid: f"{info['action_canon']}({','.join(info.get('args_canon',[]))})"
                        for pid,info in gt["prop_dict"].items()
                    }
                    rev = {v:k for k,v in mapping.items()}  # 原子命题到 prop_id 的反向映射
                    to_labels = lambda raw: [{rev.get(ap,ap) for ap in step} for step in raw]  # 将原子命题映射回 prop_id
                    good, bad = to_labels(gt["good_trace"]), to_labels(gt["bad_trace"])
                    phi = e["prediction"]
                    # 若预测是列表（分词形式），则拼接为字符串；这里用 type 比较而非 isinstance，可能不稳健
                    if type(phi) == List: phi = "".join(phi)
                    
                    # ---- build a clean LTL string ----
                    # 将预测文本正则化为 LTL 可解析的公式
                    tokens   = _tokenise(phi)                    # 将原始字符串分词为 token 列表
                    norm_str  = _normalise_tokens(tokens)        # 关键字/符号归一化
                    toks      = norm_str.split()                 # 再拆回列表以便处理
                    elim      = _elim_impl_tokens(toks)          # 蕴含消除（A-->B => (not A or B)）
                    f_str     = " ".join(elim)                   # 最终 LTL 字符串
                    try:
                        ast       = _parse(f_str)                # 解析为 AST
                        # ---- evaluate ---- 同时验证 good/bad 轨迹
                        good_sat = _eval(ast, good)              # 正例轨迹是否满足公式（应为 True）
                        bad_sat  = _eval(ast, bad)               # 反例轨迹是否满足公式（理想应为 False）

                        if good_sat:
                            ok_good += 1
                        if not bad_sat:
                            ok_bad += 1
                        if good_sat and not bad_sat:
                            ok_both += 1
                    except Exception:
                        # 解析失败统计；注意：bad_parse 未在此分支初始化，可能触发未定义错误
                        bad_parse +=1 

                    total += 1
                # 注意：此处使用了未定义的 model 变量，会导致报错
                results.append((
                    framework, lifting, model, ds, total,
                    ok_good/total, ok_bad/total, ok_both/total
                ))

    else:
        # nl2spec 结构包含模型子目录，需遍历到 model 级
        for lift_dir in fw_dir.iterdir():
            lifting = lift_dir.name
            for model_dir in lift_dir.iterdir():
                model = model_dir.name
                if '4.1-mini' not in model:
                    continue
                print(model)
                for ds in datasets:
                    file = model_dir / f"{ds}.jsonl"
                    if not file.exists():
                        continue

                    total = ok_good = ok_bad = ok_both = bad_parse = 0  # 这里初始化了 bad_parse
                    for line in tqdm(file.open(), desc=f"{framework}/{lifting}/{model}/{ds}"):
                        e = json.loads(line)
                        gt = test_entries[ds][e["id"]]

                        # rebuild prop->atom mapping
                        mapping = {
                            pid: f"{info['action_canon']}({','.join(info.get('args_canon', []))})"
                            for pid, info in gt["prop_dict"].items()
                        }
                        rev_map = {atom: pid for pid, atom in mapping.items()}  # 原子命题到 prop_id 的反向映射
                        to_labels = lambda raw: [{rev_map.get(ap, ap) for ap in step} for step in raw]  # 逆映射回复合标识
                        good, bad = to_labels(gt["good_trace"]), to_labels(gt["bad_trace"])

                        # strip any ChatGPT prefixes/suffixes
                        # 清理可能的前缀/后缀噪声，保留 LTL 主体
                        phi = e["prediction"]
                        if phi.startswith('LTL:'):
                            phi = phi[4:]
                        if phi.startswith('3. *FINAL:* '):
                            phi = phi[12:]
                        for suffix in ('*FINISH*', 'FINISH'):
                            if phi.endswith(suffix):
                                phi = phi[: -len(suffix)]

                        # ---- build a clean LTL string ----
                        tokens   = _tokenise(phi)                    # 分词
                        norm_str  = _normalise_tokens(tokens)        # 归一化
                        toks      = norm_str.split()                 # 列表化
                        elim      = _elim_impl_tokens(toks)          # 蕴含消除
                        f_str     = " ".join(elim)                   # 最终字符串
                        try:
                            ast       = _parse(f_str)                # 解析 AST
                            # ---- evaluate ---- 同时验证 good/bad 轨迹
                            good_sat = _eval(ast, good)              # 正例满足
                            bad_sat  = _eval(ast, bad)               # 反例违反（即不满足）

                            if good_sat:
                                ok_good += 1
                            if not bad_sat:
                                ok_bad += 1
                            if good_sat and not bad_sat:
                                ok_both += 1
                        except Exception:
                            bad_parse +=1                           # 解析错误计数

                        total += 1
                    # 记录该配置下的总体统计比例
                    results.append((
                        framework, lifting, model, ds, total,
                        ok_good/total, ok_bad/total, ok_both/total
                    ))

# Summarize
# 汇总所有结果为 DataFrame，便于展示和保存
columns = ["framework","lifting","model","dataset","total",
           "ok_good(%)","ok_bad(%)","ok_both(%)"]
df = pd.DataFrame(results, columns=columns)
df

nl2ltl_gpt-4.1-mini


nl2ltl/llm_masked_nl/nl2ltl_gpt-4.1-mini/search_and_rescue: 500it [00:00, 11671.85it/s]
nl2ltl/llm_masked_nl/nl2ltl_gpt-4.1-mini/traffic_light: 500it [00:00, 15798.47it/s]
nl2ltl/llm_masked_nl/nl2ltl_gpt-4.1-mini/warehouse: 500it [00:00, 15324.57it/s]


nl2ltl_gpt-4.1-mini


nl2ltl/gt_masked_nl/nl2ltl_gpt-4.1-mini/search_and_rescue: 500it [00:00, 11783.87it/s]
nl2ltl/gt_masked_nl/nl2ltl_gpt-4.1-mini/traffic_light: 500it [00:00, 13901.96it/s]
nl2ltl/gt_masked_nl/nl2ltl_gpt-4.1-mini/warehouse: 500it [00:00, 12127.87it/s]


nl2ltl_gpt-4.1-mini


...

nl2spec_gpt-4.1-mini


nl2spec/gt_masked_nl/nl2spec_gpt-4.1-mini/search_and_rescue: 500it [00:00, 8209.96it/s]
nl2spec/gt_masked_nl/nl2spec_gpt-4.1-mini/traffic_light: 500it [00:00, 8918.74it/s]
nl2spec/gt_masked_nl/nl2spec_gpt-4.1-mini/warehouse: 500it [00:00, 9547.22it/s]


nl2spec_gpt-4.1-mini


nl2spec/raw_nl/nl2spec_gpt-4.1-mini/search_and_rescue: 500it [00:00, 8847.06it/s]
nl2spec/raw_nl/nl2spec_gpt-4.1-mini/traffic_light: 500it [00:00, 9317.65it/s]
nl2spec/raw_nl/nl2spec_gpt-4.1-mini/warehouse: 500it [00:00, 10147.54it/s]
nl2tl/gt_lifting/search_and_rescue: 500it [00:00, 21624.81it/s]
nl2tl/gt_lifting/traffic_light: 500it [00:00, 20819.74it/s]
nl2tl/gt_lifting/warehouse: 500it [00:00, 20895.04it/s]
nl2tl/Qwen3-lifted-vllm/search_and_rescue: 2192it [00:00, 19225.68it/s]
nl2tl/Qwen3-lifted-vllm/traffic_light: 2196it [00:00, 19388.24it/s]
nl2tl/Qwen3-lifted-vllm/warehouse: 2238it [00:00, 19824.86it/s]
nl2tl/Qwen3-4B_r16_a32_lr5e-5/search_and_rescue: 2192it [00:00, 9283.55it/s]
nl2tl/Qwen3-4B_r16_a32_lr5e-5/traffic_light: 2196it [00:00, 8902.30it/s]
nl2tl/Qwen3-4B_r16_a32_lr5e-5/warehouse: 2238it [00:00, 9169.36it/s]
nl2tl/raw_nl/search_and_rescue: 500it [00:00, 8985.88it/s]
nl2tl/raw_nl/traffic_light: 500it [00:00, 9037.19it/s]
nl2tl/raw_nl/warehouse: 500it [00:00, 9653.93it/

,framework,lifting,model,dataset,total,ok_good(%),ok_bad(%),ok_both(%)
0,nl2ltl,llm_masked_nl,nl2ltl_gpt-4.1-mini,search_and_rescue,500,0.496000,0.588000,0.318000
1,nl2ltl,llm_masked_nl,nl2ltl_gpt-4.1-mini,traffic_light,500,0.532000,0.598000,0.362000
2,nl2ltl,llm_masked_nl,nl2ltl_gpt-4.1-mini,warehouse,500,0.448000,0.566000,0.264000
3,nl2ltl,gt_masked_nl,nl2ltl_gpt-4.1-mini,search_and_rescue,500,0.106000,0.320000,0.074000
4,nl2ltl,gt_masked_nl,nl2ltl_gpt-4.1-mini,traffic_light,500,0.618000,0.592000,0.366000
5,nl2ltl,gt_masked_nl,nl2ltl_gpt-4.1-mini,warehouse,500,0.124000,0.362000,0.098000
6,nl2ltl,raw_nl,nl2ltl_gpt-4.1-mini,search_and_rescue,500,0.616000,0.614000,0.354000
7,nl2ltl,raw_nl,nl2ltl_gpt-4.1-mini,traffic_light,500,0.646000,0.602000,0.384000
8,nl2ltl,raw_nl,nl2ltl_gpt-4.1-mini,warehouse,500,0.524000,0.586000,0.262000
9,nl2spec,llm_masked_nl,nl2spec_gpt-4.1-mini,search_and_rescue,500,0.340000,0.364000,0.210000


In [ ]:
def are_strings_similar(str1, str2, max_diff):
    """
    判断两个字符串在允许的最大字符差异数 `max_diff` 内是否可视为相似。

    参数:
        str1: 第一个字符串。
        str2: 第二个字符串。
        max_diff: 允许的最大字符差异数。

    返回:
        若差异数不超过 max_diff 返回 True，否则返回 False。
    """
    if abs(len(str1) - len(str2)) > max_diff:
        return False  # 长度差超出阈值时直接视为不相似

    diff_count = 0
    min_len = min(len(str1), len(str2))

    for i in range(min_len):
        if str1[i] != str2[i]:
            diff_count += 1  # 统计逐字符差异数
    
    diff_count += abs(len(str1) - len(str2))  # 加上多出的字符差异

    return diff_count <= max_diff  # 差异数不超过阈值即判为相似

# 示例用法
string1 = "apple"
string2 = "aplle"
max_difference = 1
result = are_strings_similar(string1, string2, max_difference)
print(f"Strings '{string1}' and '{string2}' are similar: {result}")

string3 = "banana"
string4 = "bananas"
max_difference = 1
result = are_strings_similar(string3, string4, max_difference)
print(f"Strings '{string3}' and '{string4}' are similar: {result}")

string5 = "grape"
string6 = "fruit"
max_difference = 2
result = are_strings_similar(string5, string6, max_difference)
print(f"Strings '{string5}' and '{string6}' are similar: {result}")

Strings 'apple' and 'aplle' are similar: True
Strings 'banana' and 'bananas' are similar: True
Strings 'grape' and 'fruit' are similar: False


In [ ]:
# Jupyter evaluation segment: exact-match and prop-level accuracy with fence-stripping
# 评估 grounding 任务的逐条精确匹配与命题级准确率

import json
from pathlib import Path
import pandas as pd

# Adjust these paths if needed
RESULTS_DIR = Path("grounding_eval")  # 模型输出目录
TEST_DIR    = Path("VLTL-Bench/test")  # 官方测试集

# 1. Load ground-truth prop_dicts
test_data = {}
for ds_file in TEST_DIR.glob("*.jsonl"):
    ds_name = ds_file.stem
    entries = [json.loads(line) for line in ds_file.open("r")]
    test_data[ds_name] = {entry["id"]: entry["prop_dict"] for entry in entries}  # 建立 id 到命题字典的映射

# 2. Parse model outputs and strip code fences
records = []
for model_dir in RESULTS_DIR.iterdir():
    if not model_dir.is_dir():
        continue
    model_name = model_dir.name
    for result_file in model_dir.glob("*.jsonl"):
        stem = result_file.stem              # e.g. "search_and_rescue_base"
        dataset, prompt_type = stem.rsplit("_", 1)
        responses = [json.loads(line) for line in result_file.open("r")]

        for resp in responses:
            # extract test-entry ID from custom_id: "dataset-model-entryid"
            cid = resp["custom_id"]
            entry_id = int(cid.split("-")[-1])

            # raw assistant content
            raw = resp["response"]["body"]["choices"][0]["message"]["content"]

            # strip fences and prefixes
            clean = raw.strip()
            if clean.startswith("```"):
                lines = clean.splitlines()
                # drop leading fence line
                lines = lines[1:]
                # drop trailing fence if present
                if lines and lines[-1].strip().startswith("```"):
                    lines = lines[:-1]
                clean = "\n".join(lines)
            # remove any "prop_dict:" prefix before JSON
            if clean.lstrip().startswith("prop_dict"):
                idx = clean.find("{")
                clean = clean[idx:]

            # parse JSON
            try:
                pred_dict = json.loads(clean)
            except json.JSONDecodeError:
                pred_dict = None  # 解析失败时记为空

            # ground truth
            gt_dict = test_data.get(dataset, {}).get(entry_id, {})

            # exact-match?
            exact = (pred_dict == gt_dict)

            # prop-level correctness
            total_props   = len(gt_dict)
            correct_props = 0
            if isinstance(pred_dict, dict):
                for key, val in gt_dict.items():
                    if pred_dict.get(key) == val:
                        correct_props += 1

            records.append({
                "model":         model_name,
                "dataset":       dataset,
                "prompt":        prompt_type,
                "id":            entry_id,
                "exact_match":   exact,
                "correct_props": correct_props,
                "total_props":   total_props
            })

# 3. Build DataFrame and compute accuracies
df = pd.DataFrame(records)

# a) Entry-level exact match accuracy
exact_acc = (
    df
    .groupby(["model", "prompt", "dataset"])["exact_match"]
    .mean()
    .reset_index()
    .rename(columns={"exact_match": "exact_match_accuracy"})  # 计算逐条精确匹配率
)

# b) Prop-level accuracy (micro average across all props)
prop_acc = (
    df
    .groupby(["model", "prompt", "dataset"])
    .sum()[["correct_props", "total_props"]]
    .assign(prop_accuracy=lambda x: x["correct_props"] / x["total_props"])  # 命题级微平均准确率
    .reset_index()[["model", "prompt", "dataset", "prop_accuracy"]]
)

# Merge both metrics
accuracy = exact_acc.merge(prop_acc, on=["model", "prompt", "dataset"])

accuracy

,model,prompt,dataset,exact_match_accuracy,prop_accuracy
0,3_5_turbo,base,search_and_rescue,0.342,0.569524
1,3_5_turbo,base,traffic_light,0.514,0.695441
2,3_5_turbo,base,warehouse,0.074,0.182785
3,3_5_turbo,scenario,search_and_rescue,0.636,0.766667
4,3_5_turbo,scenario,traffic_light,0.208,0.372454
5,3_5_turbo,scenario,warehouse,0.050,0.136364
6,4_1_mini,base,search_and_rescue,0.604,0.773333
7,4_1_mini,base,traffic_light,0.458,0.674103
8,4_1_mini,base,warehouse,0.078,0.237911
9,4_1_mini,scenario,search_and_rescue,0.452,0.686667
